In [0]:
create catalog sony_dev

In [0]:
create schema if not exists sony_dev.naval;

In [0]:
use catalog sony_dev;
create schema naval;

In [0]:
grant privileges on object object_name to 'users'

revoke privileges on object object_naem from 'users'

In [0]:
revoke all privileges on catalog sony_dev from `account users`

In [0]:
grant all privileges on catalog sony_dev to `account users`
--revoke all privileges on catalog sony_dev from `account users`

In [0]:
use catalog sony_dev;
use schema naval;

In [0]:

CREATE OR REPLACE TABLE heartrate_device (device_id INT, mrn STRING, name STRING, time TIMESTAMP, heartrate DOUBLE);

INSERT INTO heartrate_device VALUES
  (23, "40580129", "Nicholas Spears", "2020-02-01T00:01:58.000+0000", 54.0122153343),
  (17, "52804177", "Lynn Russell", "2020-02-01T00:02:55.000+0000", 92.5136468131),
  (37, "65300842", "Samuel Hughes", "2020-02-01T00:08:58.000+0000", 52.1354807863),
  (23, "40580129", "Nicholas Spears", "2020-02-01T00:16:51.000+0000", 54.6477014191),
  (17, "52804177", "Lynn Russell", "2020-02-01T00:18:08.000+0000", 95.033344842);
  
SELECT * FROM heartrate_device

In [0]:

CREATE OR REPLACE VIEW naval.agg_heartrate AS (
  SELECT mrn, name, MEAN(heartrate) avg_heartrate, DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);
SELECT * FROM naval.agg_heartrate

Column Level

In [0]:
CREATE OR REPLACE VIEW naval.agg_heartrate AS (
  SELECT 
  CASE WHEN
    is_account_group_member('sony_de') THEN 'REDACTED'
    ELSE mrn
  END AS mrn, 
  name, 
  MEAN(heartrate) avg_heartrate, 
  DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);

In [0]:
CREATE OR REPLACE VIEW naval.agg_heartrate AS (
  SELECT 
  CASE WHEN
    is_account_group_member('sony_de') THEN mrn
    ELSE 'Redacted'
  END AS mrn, 
  name, 
  MEAN(heartrate) avg_heartrate, 
  DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);

In [0]:
select * from naval.agg_heartrate

In [0]:
create or replace view naval.heartrate 
as 
SELECT * FROM heartrate_device  
where
CASE
    WHEN is_account_group_member('account users') THEN device_id<30
    ELSE True
  END;

In [0]:
select * from naval.heartrate

In [0]:

CREATE OR REPLACE FUNCTION sony_dev.naval.datamask(x STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(x) - 2), RIGHT(x, 2)
); 
     

In [0]:
CREATE OR REPLACE FUNCTION sony_dev.naval.mask_email(email STRING)
  RETURNS STRING
  RETURN CONCAT(REPEAT("*", LENGTH(email) - POSITION('@', email)), SUBSTR(email, POSITION('@', email)));

In [0]:
select sony_dev.naval.mask_email('naval@gmail.com') as email

In [0]:

CREATE OR REPLACE VIEW naval.agg_heartrate AS (
  SELECT 
  CASE WHEN
    is_account_group_member('account users') THEN naval.datamask(mrn)
    ELSE mrn
  END AS mrn, 
  name, 
  MEAN(heartrate) avg_heartrate, 
  DATE_TRUNC("DD", time) date
  FROM heartrate_device
  GROUP BY mrn, name, DATE_TRUNC("DD", time)
);

In [0]:
select * from naval.agg_heartrate